# RECOMMENDATION SYSTEM

In [ ]:
import pandas as pd
import pickle
import numpy as np
import pandas as pd

## Paso 1: Definimos el problema de recomendación

### ¿Qué se quiere recomendar?

#### Trayectorias o características personales y laborales que una persona podría cambiar para aumentar su probabilidad de ganar más de $50K. Es decir, el sistema recomendará mejoras en el perfil de un usuario (educación, tipo de ocupación, horas trabajadas, etc.) que aumenten la probabilidad de tener ingresos más altos

### ¿Quién será el usuario?

#### El usuario será un perfil extraído del dataset que tenga ingresos menores a 50K (como ejemplos reales)

### ¿Qué variables definen el perfil del usuario?

#### Todas las variables predictoras que se usaron en el modelo XGBoost o variables que más correlacion tengan con el target



## Paso 2: Construimos el sistema de recomendación

In [2]:
with open('../data/processed/13_df_eda.pkl', 'rb') as f:
    df = pickle.load(f)

In [3]:
with open('../models/13-opt-xgb-model.pkl', 'rb') as f:
    model = pickle.load(f)

In [16]:
df.head()

,age,education.num,capital.gain,capital.loss,hours.per.week,income_bin,workclass_f,marital.status_f,occupation_f,relationship_f,race_f,sex_f,native.country_f
1,82,9,0,4356,18,0,0.0,0,0.0,0,0,0,0.0
3,54,4,0,3900,40,0,0.0,1,1.0,1,0,0,0.0
4,41,10,0,3900,40,0,0.0,2,2.0,2,0,0,0.0
5,34,9,0,3770,45,0,0.0,1,3.0,1,0,0,0.0
6,38,6,0,3770,40,0,0.0,2,4.0,1,0,1,0.0


In [ ]:
# =========================
# Configuración
# =========================
TARGET = 'income_bin'                 
EXCLUDE_COLS = [TARGET]
# Columnas numéricas continuas (probamos cuantiles razonables)
NUMERIC_CONT = ['age', 'education.num', 'hours.per.week', 'capital.gain', 'capital.loss']

# Columnas codificadas de categorías (terminan en _f)
# Si no estás seguro, las detectamos automáticamente por sufijo:
CAT_SUFFIX = '_f'

# Cuantiles a simular en numéricas (puedes ampliar)
NUM_QUANTILES = [0.25, 0.50, 0.75, 0.90]

# Top categorías más frecuentes a probar (por columna categórica codificada)
TOP_N_CATS = 5

# =========================
# 1) Elegir un usuario con income < 50K
# =========================
candidatos = df[df[TARGET] == 0]
assert len(candidatos) > 0, "No hay usuarios con income_bin == 0."

# Elige uno al azar (puedes fijar un índice concreto si quieres)
usuario_idx = candidatos.sample(1, random_state=42).index[0]
usuario = df.loc[usuario_idx].copy()

# Columnas de entrada EXACTAS del modelo (mismas que al entrenar)
X_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

# =========================
# 2) Probabilidad actual con el modelo
# =========================
X_user = usuario[X_cols].to_frame().T
base_prob = float(model.predict_proba(X_user)[:, 1][0])

# =========================
# 3) Preparar grillas de simulación
# =========================
# Detectar columnas categóricas codificadas por sufijo (_f) que estén en X_cols
CAT_COLS = [c for c in X_cols if c.endswith(CAT_SUFFIX)]
# Ver cuáles de NUMERIC_CONT realmente existen en X_cols
NUM_COLS = [c for c in NUMERIC_CONT if c in X_cols]

# Grilla para numéricas: cuantiles + valor actual (evita duplicados)
numeric_grid = {}
for col in NUM_COLS:
    s = df[col].dropna()
    if s.nunique() <= 1:
        numeric_grid[col] = [usuario[col]]
        continue
    qs = s.quantile(NUM_QUANTILES).values.tolist()
    # Mantener tipo entero si la columna parece entera
    if pd.api.types.is_integer_dtype(s):
        qs = sorted(set(int(round(v)) for v in qs))
    else:
        qs = sorted(set(float(v) for v in qs))
    qs.append(usuario[col])
    numeric_grid[col] = sorted(set(qs))

# Grilla para categóricas: TOP_N categorías más frecuentes + valor actual
categorical_grid = {}
for col in CAT_COLS:
    top_vals = df[col].value_counts(dropna=True).head(TOP_N_CATS).index.tolist()
    # Incluye el valor actual si no está
    if usuario[col] not in top_vals:
        top_vals.append(usuario[col])
    # Convertir a tipos "limpios" (muchas *_f vienen como float .0 -> pasar a int si aplica)
    vals = []
    for v in top_vals:
        if pd.notna(v) and float(v).is_integer():
            vals.append(int(v))
        else:
            vals.append(v)
    # Asegurar que el valor actual esté exacto
    if float(usuario[col]).is_integer():
        usuario_val = int(usuario[col])
    else:
        usuario_val = usuario[col]
    categorical_grid[col] = sorted(set(vals), key=lambda x: (isinstance(x, float), x))

# =========================
# 4) Simular cambios y construir recomendaciones
# =========================
mejoras = []  # (col, actual, sugerido, prob_nueva, delta)

# a) Cambios en numéricas (uno a uno)
for col in NUM_COLS:
    mejor_prob = base_prob
    mejor_val = usuario[col]
    for val in numeric_grid[col]:
        if val == usuario[col]:
            continue
        perfil = usuario.copy()
        perfil[col] = val
        X_tmp = perfil[X_cols].to_frame().T
        p = float(model.predict_proba(X_tmp)[:, 1][0])
        if p > mejor_prob:
            mejor_prob = p
            mejor_val = val
    if mejor_val != usuario[col]:
        mejoras.append((col, usuario[col], mejor_val, mejor_prob, mejor_prob - base_prob))

# b) Cambios en categóricas codificadas (uno a uno)
for col in CAT_COLS:
    mejor_prob = base_prob
    mejor_val = usuario[col]
    # Normalizar valor actual para comparación (int si es x.0)
    actual_val = int(usuario[col]) if float(usuario[col]).is_integer() else usuario[col]
    for val in categorical_grid[col]:
        if val == actual_val:
            continue
        perfil = usuario.copy()
        perfil[col] = val
        X_tmp = perfil[X_cols].to_frame().T
        p = float(model.predict_proba(X_tmp)[:, 1][0])
        if p > mejor_prob:
            mejor_prob = p
            mejor_val = val
    if mejor_val != actual_val:
        mejoras.append((col, actual_val, mejor_val, mejor_prob, mejor_prob - base_prob))

# Ordenar por mayor incremento de probabilidad
mejoras_ordenadas = sorted(mejoras, key=lambda x: x[-1], reverse=True)

# =========================
# Resultados
# =========================
print(f"Usuario elegido index={usuario_idx} (income_bin=0)")
print(f"Probabilidad actual de >50K: {base_prob:.4f}\n")

if not mejoras_ordenadas:
    print("No se encontraron mejoras univariantes con las rejillas probadas.")
else:
    print("📌 Mejores recomendaciones para ganar >50k:")
    for col, actual, sugerido, p_nueva, delta in mejoras_ordenadas[:10]:
        print(f"- {col}: {actual} → {sugerido} | prob nueva: {p_nueva:.4f} (Δ={delta:.4f})")




Usuario elegido index=11343 (income_bin=0)
Probabilidad actual de >50K: 0.0022

📌 Mejores recomendaciones para ganar >50k:
- age: 22.0 → 47 | prob nueva: 0.0656 (Δ=0.0634)
- relationship_f: 2 → 5 | prob nueva: 0.0318 (Δ=0.0296)
- marital.status_f: 3 → 4 | prob nueva: 0.0132 (Δ=0.0110)
- hours.per.week: 40.0 → 55 | prob nueva: 0.0074 (Δ=0.0052)
- workclass_f: 0 → 5 | prob nueva: 0.0071 (Δ=0.0049)
- occupation_f: 9 → 2 | prob nueva: 0.0059 (Δ=0.0037)
- native.country_f: 0 → 7 | prob nueva: 0.0022 (Δ=0.0000)


## Paso 3: Pruebas con casos simulados

In [ ]:


# ==========================================
# Configuración general
# ==========================================
TARGET = 'income_bin'
EXCLUDE_COLS = [TARGET]  
CAT_SUFFIX = '_f'                  # sufijo de columnas categóricas factorizadas
NUM_QUANTILES = [0.25, 0.5, 0.75, 0.9]
TOP_N_CATS = 5                     # top categorías a explorar por columna *_f
TOP_K_RECOMENDACIONES = 5          # cuántas recomendaciones mostrar por perfil

# Si quieres fijar qué numéricas explorar explícitamente, ponlas aquí; si no, se detectan por dtype
NUMERIC_CONT_EXPL = ['age', 'education.num', 'hours.per.week', 'capital.gain', 'capital.loss']

# Columnas de entrada EXACTAS del modelo
X_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]

# Detectar columnas *_f (categóricas codificadas) y numéricas presentes
CAT_COLS = [c for c in X_COLS if c.endswith(CAT_SUFFIX)]
NUM_COLS = [c for c in (NUMERIC_CONT_EXPL if NUMERIC_CONT_EXPL else X_COLS) if c in X_COLS and not c.endswith(CAT_SUFFIX)]

# Precalcular grillas (rejillas) de valores razonables por columna a partir de df
def build_grids(df):
    numeric_grid = {}
    for col in NUM_COLS:
        s = df[col].dropna()
        if s.nunique() <= 1:
            numeric_grid[col] = [s.iloc[0] if len(s) else 0]
            continue
        qs = s.quantile(NUM_QUANTILES).values.tolist()
        # Mantener enteros para columnas enteras
        if pd.api.types.is_integer_dtype(s):
            qs = sorted(set(int(round(v)) for v in qs))
        else:
            qs = sorted(set(float(v) for v in qs))
        numeric_grid[col] = qs

    categorical_grid = {}
    for col in CAT_COLS:
        top_vals = df[col].value_counts(dropna=True).head(TOP_N_CATS).index.tolist()
        # Normalizar (muchas *_f vienen como 0.0 → int)
        vals = []
        for v in top_vals:
            if pd.notna(v) and float(v).is_integer():
                vals.append(int(v))
            else:
                vals.append(v)
        categorical_grid[col] = sorted(set(vals), key=lambda x: (isinstance(x, float), x))
    return numeric_grid, categorical_grid

NUMERIC_GRID, CATEGORICAL_GRID = build_grids(df)

# Utilidad: completa un perfil hipotético con defaults (medianas/modas) y lo ajusta a X_COLS
def normalize_profile(profile_dict, df, x_cols):
    # Base con medianas (num) y modas (cat *_f)
    base = {}
    for col in x_cols:
        s = df[col].dropna()
        if col in CAT_COLS:
            # usar moda como default
            val = s.mode().iloc[0] if not s.mode().empty else (int(s.median()) if len(s) else 0)
            # normalizar a int si es x.0
            base[col] = int(val) if float(val).is_integer() else val
        else:
            # usar mediana para numéricas
            val = s.median() if len(s) else 0
            # si aparenta entero, castear a int
            base[col] = int(val) if pd.api.types.is_integer_dtype(s) or float(val).is_integer() else float(val)

    # Sobrescribir con el perfil del usuario
    for k, v in profile_dict.items():
        if k in x_cols:
            if k in CAT_COLS:
                # normalizar categóricas a int si procede
                v2 = int(v) if pd.notna(v) and float(v).is_integer() else v
                base[k] = v2
            else:
                # asegurar tipo numérico
                base[k] = float(v) if isinstance(v, (float, int, np.floating, np.integer)) else base[k]

    return pd.Series(base)

# Simulamos cambios univariantes para un perfil dado y devuelve mejores recomendaciones
def recomendar_para_perfil(perfil_series, model, x_cols, numeric_grid, categorical_grid, top_k=TOP_K_RECOMENDACIONES):
    X_user = perfil_series[x_cols].to_frame().T
    base_prob = float(model.predict_proba(X_user)[:, 1][0])

    mejoras = []  # (col, actual, sugerido, prob_nueva, delta)

    # a) Explorar numéricas
    for col in NUM_COLS:
        actual = perfil_series[col]
        mejor_prob = base_prob
        mejor_val = actual
        # Probamos rejilla + valor actual por si falta
        candidates = sorted(set(numeric_grid.get(col, [] ) + [actual]))
        for val in candidates:
            if val == actual:
                continue
            perfil_tmp = perfil_series.copy()
            perfil_tmp[col] = val
            p = float(model.predict_proba(perfil_tmp[x_cols].to_frame().T)[:, 1][0])
            if p > mejor_prob:
                mejor_prob = p
                mejor_val = val
        if mejor_val != actual:
            mejoras.append((col, actual, mejor_val, mejor_prob, mejor_prob - base_prob))

    # b) Explorar categóricas *_f
    for col in CAT_COLS:
        actual = perfil_series[col]
        actual_norm = int(actual) if float(actual).is_integer() else actual
        mejor_prob = base_prob
        mejor_val = actual_norm
        candidates = sorted(set(categorical_grid.get(col, [] ) + [actual_norm]), key=lambda x: (isinstance(x, float), x))
        for val in candidates:
            if val == actual_norm:
                continue
            perfil_tmp = perfil_series.copy()
            perfil_tmp[col] = val
            p = float(model.predict_proba(perfil_tmp[x_cols].to_frame().T)[:, 1][0])
            if p > mejor_prob:
                mejor_prob = p
                mejor_val = val
        if mejor_val != actual_norm:
            mejoras.append((col, actual_norm, mejor_val, mejor_prob, mejor_prob - base_prob))

    mejoras_ordenadas = sorted(mejoras, key=lambda x: x[-1], reverse=True)[:top_k]
    return base_prob, mejoras_ordenadas

# ==========================================
# Definir perfiles simulados (hipotéticos)
# ==========================================
perfiles_simulados = [
    # Perfil 1: joven con educación media y pocas horas
    {
        'age': 28,
        'education.num': 10,        # Some-college aprox
        'hours.per.week': 35,
        'workclass_f': 0,           # ajusta según tu codificación
        'occupation_f': 3,
        'relationship_f': 1,
        'sex_f': 1
    },
    # Perfil 2: adulto con HS-grad, sin ganancias de capital
    {
        'age': 45,
        'education.num': 9,         # HS-grad
        'hours.per.week': 40,
        'capital.gain': 0,
        'capital.loss': 0,
        'workclass_f': 2,
        'occupation_f': 4,
        'relationship_f': 2,
        'sex_f': 0
    },
    # Perfil 3: alta educación pero pocas horas
    {
        'age': 38,
        'education.num': 14,        # Masters
        'hours.per.week': 30,
        'workclass_f': 0,
        'occupation_f': 2,
        'relationship_f': 1,
        'sex_f': 1
    },
    # Perfil 4: mucha experiencia (edad alta), horas altas
    {
        'age': 58,
        'education.num': 10,
        'hours.per.week': 50,
        'workclass_f': 2,
        'occupation_f': 1,
        'relationship_f': 0,
        'sex_f': 0
    },
    # Perfil 5: migrante hipotético con ocupación operativa
    {
        'age': 34,
        'education.num': 12,        
        'hours.per.week': 40,
        'native.country_f': 3,      
        'occupation_f': 6,
        'workclass_f': 0,
        'relationship_f': 1,
        'sex_f': 1
    },
]

# ==========================================
# Ejecutar recomendaciones para cada perfil
# ==========================================
for i, perfil in enumerate(perfiles_simulados, start=1):
    perfil_series = normalize_profile(perfil, df, X_COLS)
    base_prob, mejoras = recomendar_para_perfil(perfil_series, model, X_COLS, NUMERIC_GRID, CATEGORICAL_GRID, top_k=TOP_K_RECOMENDACIONES)

    print("="*70)
    print(f"🧑‍💻 Perfil simulado #{i}")
    print("Entrada (parcial):", {k: perfil.get(k, 'default') for k in perfil.keys()})
    print(f"Probabilidad actual >50K: {base_prob:.4f}")

    if not mejoras:
        print("No se encontraron mejoras univariantes con las rejillas probadas.")
    else:
        print("📌 Recomendaciones (top cambios univariantes por incremento de probabilidad):")
        for col, actual, sugerido, p_nueva, delta in mejoras:
            print(f" - {col}: {actual} → {sugerido} | prob nueva: {p_nueva:.4f} (Δ={delta:.4f})")

     # Aplicar la mejor y mostrar la nueva probabilidad
    if mejoras:
        col_b, actual_b, sugerido_b, p_nueva_b, delta_b = mejoras[0]
        perfil_best = perfil_series.copy()
        perfil_best[col_b] = sugerido_b
        p_best = float(model.predict_proba(perfil_best[X_COLS].to_frame().T)[:, 1][0])
        print("✅ Recomendación prioritaria:")
        print(f"   Cambiar {col_b} de {actual_b} a {sugerido_b} → prob >50K ≈ {p_best:.4f}")



🧑‍💻 Perfil simulado #1
Entrada (parcial): {'age': 28, 'education.num': 10, 'hours.per.week': 35, 'workclass_f': 0, 'occupation_f': 3, 'relationship_f': 1, 'sex_f': 1}
Probabilidad actual >50K: 0.0443
📌 Recomendaciones (top cambios univariantes por incremento de probabilidad):
 - hours.per.week: 35.0 → 55 | prob nueva: 0.2000 (Δ=0.1557)
 - relationship_f: 1 → 5 | prob nueva: 0.1788 (Δ=0.1345)
 - age: 28.0 → 47 | prob nueva: 0.1045 (Δ=0.0601)
 - occupation_f: 3 → 2 | prob nueva: 0.0818 (Δ=0.0375)
 - sex_f: 1 → 0 | prob nueva: 0.0745 (Δ=0.0301)
✅ Recomendación prioritaria:
   Cambiar hours.per.week de 35.0 a 55 → prob >50K ≈ 0.2000
🧑‍💻 Perfil simulado #2
Entrada (parcial): {'age': 45, 'education.num': 9, 'hours.per.week': 40, 'capital.gain': 0, 'capital.loss': 0, 'workclass_f': 2, 'occupation_f': 4, 'relationship_f': 2, 'sex_f': 0}
Probabilidad actual >50K: 0.2010
📌 Recomendaciones (top cambios univariantes por incremento de probabilidad):
 - relationship_f: 2 → 5 | prob nueva: 0.5757 (Δ=